### PennCATH GWAS

Continues `gwas03b_penchat.ipynb` (QC). Input is the QC-passing fileset
`data/PennCATH/results/penncath_qc.{bed,bim,fam}` — 696,660 variants ×
1,382 samples, 923 cases / 459 controls, build **GRCh37/hg19**.

Functions live in `src/scripts/assoc_penncath_lib.py`.

### Before start

```Bash
conda activate genome_env
```

In [1]:
import os, sys, csv
import numpy as np
import pandas as pd
from pathlib import Path

ROOT0 = Path("/home/flavio/uv/gwas/")
ROOT_SRC = ROOT0 / "src"

sys.path.insert(0, str(ROOT_SRC))
sys.path.append(str(ROOT_SRC))

from libs.Basic import create_dir, pdwritecsv, pdreadcsv

### Point gwaslib at genome_env

Same block as the QC notebook.

In [2]:
os.environ["GWAS_CONDA_PREFIX"] = "/home/flavio/miniforge3/envs/genome_env"
os.environ["PLINK"]  = "/home/flavio/miniforge3/envs/genome_env/bin/plink"
os.environ["PLINK2"] = "/home/flavio/miniforge3/envs/genome_env/bin/plink2"

from scripts import gwaslib

gwaslib.PLINK  = os.environ["PLINK"]
gwaslib.PLINK2 = os.environ["PLINK2"]

from scripts.assoc_penncath_lib import *

In [3]:
gwaslib.setup_logging("assoc_penncath")

for directory in gwaslib.ALL_DIRS:
    directory.mkdir(parents=True, exist_ok=True)

gwaslib.require(gwaslib.PLINK)
gwaslib.require(gwaslib.PLINK2)

15:05:32 INFO    logging to /home/flavio/uv/gwas/logs/assoc_penncath.log


'/home/flavio/miniforge3/envs/genome_env/bin/plink2'

In [4]:
QC_STEM, PCA_STEM

(PosixPath('/home/flavio/uv/gwas/data/PennCATH/results/penncath_qc'),
 PosixPath('/home/flavio/uv/gwas/data/PennCATH/results/penncath_pca'))

In [5]:
gwaslib.count_bfile(QC_STEM)

(696660, 1382)

### 1. Covariates

sex + age + PC1–4.

PCs 1–4 only: the QC eigenvalues were 6.24, 2.19, 1.85, 1.69, then flat at
~1.35, so PC5 onward is noise.

The lipids (`tg`, `hdl`, `ldl`) are **excluded on purpose** — they sit on the
causal pathway from genotype to CAD, so conditioning on them would attenuate
the effects we are testing, and they are missing for 92–119 samples.

In [8]:
filename_covar = build_assoc_covar(n_pcs=4)
filename_covar

15:07:38 INFO    covariates for 1382 samples: sex, age, PC1, PC2, PC3, PC4


PosixPath('/home/flavio/uv/gwas/data/PennCATH/interim/penncath_assoc.covar')

In [11]:
df_cov = pdreadcsv(filename_covar)
print(df_cov.shape)

df_cov.head(3)

(1382, 8)


,FID,IID,PC1,PC2,PC3,PC4,sex,age
0,10002,1,0.012013,-0.013698,-0.011841,-0.021991,1,60
1,10004,1,-0.012775,-0.005919,0.011620,0.034670,2,50
2,10005,1,-0.016200,0.007924,-0.030831,0.009019,1,55


## 2. Association test

`plink2 --glm firth-fallback hide-covar` — logistic regression on allele
dosage. Firth fallback protects against separation in sparse tables;
`--covar-variance-standardize` only rescales covariates, not the SNP effect.

In [13]:
glm = run_glm(filename_covar, threads=14)
glm

15:08:51 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink2 --bfile /home/flavio/uv/gwas/data/PennCATH/results/penncath_qc --covar /home/flavio/uv/gwas/data/PennCATH/interim/penncath_assoc.covar --covar-variance-standardize --glm firth-fallback hide-covar --ci 0.95 --threads 14 --out /home/flavio/uv/gwas/data/PennCATH/results/penncath_assoc
15:09:02 INFO    ok (10.8s)
15:09:02 INFO    association table: penncath_assoc.PHENO1.glm.logistic.hybrid


PosixPath('/home/flavio/uv/gwas/data/PennCATH/results/penncath_assoc.PHENO1.glm.logistic.hybrid')

In [16]:
df = load_glm(glm)
df.shape, np.array(df.columns)

15:09:33 INFO    696660 variants tested, 0 with a non-'.' ERRCODE
15:09:33 INFO    Firth fallback used for 0 variants


((696660, 19),
 array(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'PROVISIONAL_REF?', 'A1',
        'OMITTED', 'A1_FREQ', 'FIRTH?', 'TEST', 'OBS_CT', 'OR',
        'LOG(OR)_SE', 'L95', 'U95', 'Z_STAT', 'P', 'ERRCODE'], dtype=object))

In [15]:
df.head(3)

,CHROM,POS,ID,REF,ALT,PROVISIONAL_REF?,A1,OMITTED,A1_FREQ,FIRTH?,TEST,OBS_CT,OR,LOG(OR)_SE,L95,U95,Z_STAT,P,ERRCODE
0,1,721290,rs12565286,C,G,Y,G,C,0.051977,N,ADD,1366,1.05520,0.213563,0.694304,1.60369,0.251593,0.801356,.
1,1,752566,rs3094315,T,C,Y,C,T,0.177997,N,ADD,1368,1.01377,0.117798,0.804764,1.27705,0.116076,0.907592,.
2,1,761732,rs2286139,T,C,Y,C,T,0.136701,N,ADD,1346,0.92593,0.135071,0.710567,1.20657,-0.569750,0.568848,.


### 3. Is the null calibrated?

λ_GC compares the median observed χ² to its null expectation. 1.0 means no
inflation; much above ~1.05 at this sample size would point to residual
population structure or relatedness.

In [17]:
lam = lambda_gc(df["P"])
f"lambda_GC = {lam:.4f}"

'lambda_GC = 1.0095'

In [20]:
f"Params: GENOME_WIDE = {GENOME_WIDE}, SUGGESTIVE = {SUGGESTIVE}"

'Params: GENOME_WIDE = 5e-08, SUGGESTIVE = 1e-05'

In [21]:
n_gw   = int((df["P"] < GENOME_WIDE).sum())
n_sugg = int((df["P"] < SUGGESTIVE).sum())

f"{n_gw} variants below 5e-8, {n_sugg} below 1e-5"

'0 variants below 5e-8, 12 below 1e-5'

In [22]:
top_hits(df)

,CHROM,POS,ID,A1,A1_FREQ,OBS_CT,OR,L95,U95,Z_STAT,P,FIRTH?
0,5,107157011,rs4957723,C,0.280753,1382,1.812700,1.458020,2.253670,5.35416,8.595610e-08,N
1,5,107159013,rs17160628,T,0.280956,1381,1.800310,1.448450,2.237650,5.29909,1.163810e-07,N
2,9,22072301,rs9632884,G,0.442190,1306,0.616312,0.507402,0.748599,-4.87851,1.068930e-06,N
3,5,107139663,rs10515379,T,0.265387,1381,1.746170,1.395760,2.184560,4.87765,1.073560e-06,N
4,9,22081850,rs6475606,G,0.454052,1382,0.637210,0.531025,0.764628,-4.84541,1.263520e-06,N
5,9,22125503,rs1333049,G,0.476483,1382,0.640561,0.532504,0.770544,-4.72518,2.299070e-06,N
6,4,188698095,rs7683009,G,0.482520,1373,0.634929,0.525371,0.767333,-4.70044,2.596060e-06,N
7,4,190300221,rs4863155,G,0.347515,1328,0.613672,0.500043,0.753122,-4.67381,2.956670e-06,N
8,9,22098574,rs4977574,A,0.462346,1381,0.645161,0.536162,0.776320,-4.64143,3.460080e-06,N
9,9,22098619,rs2891168,A,0.463097,1382,0.647949,0.538577,0.779531,-4.60034,4.218010e-06,N


### 4. Independent loci

One causal variant drags its whole LD block over the threshold, so the raw hit
count overstates the number of findings. Clumping keeps the lead SNP per block.

`--clump` runs under PLINK **1.9** — it is absent from the plink2 build here.

In [25]:
clumped = run_clump(glm)
df_cl = load_clumps(clumped)
print(df_cl.shape)
print(df_cl.columns)

cols = ["CHR", "BP", "SNP", "P", "TOTAL"]
df_cl

15:12:42 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/results/penncath_qc --clump /home/flavio/uv/gwas/data/PennCATH/results/penncath_assoc.PHENO1.glm.logistic.hybrid --clump-snp-field ID --clump-field P --clump-p1 1e-05 --clump-p2 0.001 --clump-r2 0.1 --clump-kb 500 --out /home/flavio/uv/gwas/data/PennCATH/results/penncath_clump
15:12:42 INFO    ok (0.3s)
(4, 12)
Index(['CHR', 'F', 'SNP', 'BP', 'P', 'TOTAL', 'NSIG', 'S05', 'S01', 'S001',
       'S0001', 'SP2'],
      dtype='object')


,CHR,F,SNP,BP,P,TOTAL,NSIG,S05,S01,S001,S0001,SP2
0,5,1,rs4957723,107157011,8.600000e-08,25,3,9,8,1,4,"rs10515379(1),rs7700475(1),rs17160628(1),rs770..."
1,9,1,rs9632884,22072301,1.070000e-06,23,1,6,7,1,8,"rs8181047(1),rs10757270(1),rs6475606(1),rs1075..."
2,4,1,rs7683009,188698095,2.600000e-06,4,4,0,0,0,0,NONE
3,4,1,rs4863155,190300221,2.960000e-06,22,5,7,5,5,0,"rs6553297(1),rs11132630(1),rs4075236(1),rs7689..."


In [26]:
pdwritecsv(df_cl, "penncath_independent_loci.csv", gwaslib.ROOT_RESULTS)

True

### 5. Gene context

Annotation queries the **GRCh37** Ensembl endpoint — PennCATH is hg19, and
GRCh38 positions differ by tens of kb at these loci.

A GWAS hit is not evidence that the nearest gene is causal: most lead variants
are non-coding and act through regulatory elements that can skip over
intervening genes. This is a locus label, not a causal assignment.

In [29]:
df_leads = df[df["ID"].isin(df_cl["SNP"])][
    ["CHROM", "POS", "ID", "A1", "A1_FREQ", "OR", "L95", "U95", "P"]].sort_values("P")

df_ann = annotate_loci(df_leads)

cols = ["ID", "CHROM", "POS", "P", "gene_at_variant", "protein_coding_within_window"]
df_ann[cols]

,ID,CHROM,POS,P,gene_at_variant,protein_coding_within_window
0,rs4957723,5,107157011,8.595610e-08,-,"EFNA5 (150 kb), FBXL17 (37 kb)"
1,rs9632884,9,22072301,1.068930e-06,CDKN2B-AS1,"MTAP (140 kb), RP11-145E5.5 (39 kb), C9orf53 (..."
2,rs7683009,4,188698095,2.596060e-06,-,-
3,rs4863155,4,190300221,2.956670e-06,-,-


### 6. Power — why nothing reached genome-wide significance

Wald-based power for a per-allele log-odds, at α = 5e-8 with n = 1,382 and
66.8 % cases.

In [32]:
n, case_frac = power_params(df)

df_pw = pd.DataFrame({"MAF": [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]})

df_pw["OR_detectable_80pct_power"] = [round(detectable_or(m, n=n, case_frac=case_frac), 6) for m in df_pw["MAF"]]

df_pw["power_for_published_9p21_OR_1.27"] = [round(power_for(1.27, m, n=n, case_frac=case_frac), 6) for m in df_pw["MAF"]]

print(f"n = {n}, cases = {case_frac:.1%}")
df_pw

n = 1382, cases = 66.8%


,MAF,OR_detectable_80pct_power,power_for_published_9p21_OR_1.27
0,0.05,3.209471,0.000016
1,0.10,2.333003,0.000119
2,0.20,1.887715,0.001021
3,0.30,1.741234,0.003079
4,0.40,1.679969,0.005356
5,0.50,1.662452,0.006348


In [33]:
# 9p21: our estimate vs the published effect
row = df[df["ID"] == "rs1333049"].iloc[0]

(f"rs1333049  our OR (risk allele C) = {1 / row['OR']:.2f}, p = {row['P']:.2e}",
 "published OR = 1.27 (GCST001079, p = 7e-58)",
 f"power to detect OR 1.27 at MAF 0.485 = "
 f"{power_for(1.27, 0.485, n=n, case_frac=case_frac):.3f}")

('rs1333049  our OR (risk allele C) = 1.56, p = 2.30e-06',
 'published OR = 1.27 (GCST001079, p = 7e-58)',
 'power to detect OR 1.27 at MAF 0.485 = 0.006')

The discovery estimate is inflated relative to the published 1.27. That is
expected rather than an error: an effect only crosses a threshold in a small
study when chance pushes it upward (winner's curse). The locus and direction
replicate; the magnitude should not be quoted from this cohort.

#3# 7. Figures

Manhattan + QQ, and regional plots for the two strongest loci. The plotting
code lives in the next cell rather than the library because it is specific to
this write-up.

In [34]:
ld_specs = [("rs9632884", 9, 21_572_301, 22_572_301, "ld_9p21"),
            ("rs4957723", 5, 106_657_011, 107_657_011, "ld_5q21")]

for snp, chrom, start, end, stem in ld_specs:
    gwaslib.plink(["--bfile", QC_STEM, "--r2", "--ld-snp", snp,
                   "--ld-window-kb", 1000, "--ld-window", 99999,
                   "--ld-window-r2", 0, "--chr", chrom,
                   "--from-bp", start, "--to-bp", end],
                  out=gwaslib.ROOT_INTERIM / stem)

15:17:34 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/results/penncath_qc --r2 --ld-snp rs9632884 --ld-window-kb 1000 --ld-window 99999 --ld-window-r2 0 --chr 9 --from-bp 21572301 --to-bp 22572301 --out /home/flavio/uv/gwas/data/PennCATH/interim/ld_9p21
15:17:34 INFO    ok (0.1s)
15:17:34 INFO    $ /home/flavio/miniforge3/envs/genome_env/bin/plink --bfile /home/flavio/uv/gwas/data/PennCATH/results/penncath_qc --r2 --ld-snp rs4957723 --ld-window-kb 1000 --ld-window 99999 --ld-window-r2 0 --chr 5 --from-bp 106657011 --to-bp 107657011 --out /home/flavio/uv/gwas/data/PennCATH/interim/ld_5q21
15:17:34 INFO    ok (0.1s)


In [36]:
labels = {"rs9632884": "9p21  $\\it{CDKN2B}$-$\\it{AS1}$",
          "rs4957723": "5q21  near $\\it{FBXL17}$"}

fig_manhattan = make_manhattan_qq(df, df_cl, labels=labels)

fig_loci = make_locus_plots(df, [
    ("rs9632884", "ld_9p21", "9p21.3 \u2014 replicates the known CAD locus"),
    ("rs4957723", "ld_5q21", "5q21.1 \u2014 not in the GWAS Catalog"),
])

fig_manhattan, fig_loci

15:18:09 INFO    wrote /home/flavio/uv/gwas/data/PennCATH/figures/penncath_manhattan_qq.png
15:18:09 INFO    wrote /home/flavio/uv/gwas/data/PennCATH/figures/penncath_loci.png


(PosixPath('/home/flavio/uv/gwas/data/PennCATH/figures/penncath_manhattan_qq.png'),
 PosixPath('/home/flavio/uv/gwas/data/PennCATH/figures/penncath_loci.png'))

![manhattan](/home/flavio/uv/gwas/data/PennCATH/figures/penncath_manhattan_qq.png)

![loci](/home/flavio/uv/gwas/data/PennCATH/figures/penncath_loci.png)

PosixPath('/home/flavio/uv/gwas/data/PennCATH/figures/penncath_manhattan_qq.png')

## Summary

| Lead SNP | Locus | OR (risk allele) | p | Replicated? |
|---|---|---|---|---|
| rs4957723 | 5q21.1 | 1.81 | 8.6e-8 | no catalogue entry |
| rs9632884 | 9p21.3 *CDKN2B-AS1* | 1.62 | 1.1e-6 | **yes** — carotid atherosclerosis p=5e-12 |
| rs7683009 | 4q35.1 | 1.57 | 2.6e-6 | no catalogue entry |
| rs4863155 | 4q35.2 | 1.63 | 3.0e-6 | no catalogue entry |

No locus reaches 5e-8. λ_GC = 1.009, so the null is calibrated and this is a
power limit. 9p21 is the one locus with independent support; `rs1333049` in the
same clump is catalogued for coronary artery disorder at p = 2e-192.

Not done: conditional analysis, imputation, sex-stratified tests, gene-based or
pathway analysis, polygenic scoring.